# Notebook 39 — Optuna : Stratégie 3 Curriculum Learning (Imagewoof)

Recherche automatique des hyperparamètres via **Optuna** (20 trials, TPE sampler).

## Hyperparamètres optimisés
| Paramètre | Espace | Défaut |
|-----------|--------|--------|
| `lr`          | log-uniform [1e-4, 1e-2] | 1e-3 |
| `ep_per_phase`| int [3, 8] — durée de chaque phase | 5 |
| `ratio_mid1`  | uniform [0.10, 0.40] — ratio HF phase 2 | 0.25 |
| `ratio_mid2`  | uniform [0.45, 0.85] — ratio HF phase 3 | 0.60 |

Total epochs = `4 × ep_per_phase` (12 à 32 époques selon le trial).

In [ ]:
import os, json, time, random, statistics
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, Sampler
from torchvision import datasets, models
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _OPTUNA_AVAILABLE = True
except ImportError:
    _OPTUNA_AVAILABLE = False
    print('optuna non installe: pip install optuna')

try:
    import wandb; _WANDB_AVAILABLE = True
except ImportError:
    _WANDB_AVAILABLE = False

SEED = 42; random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Appareil: {device}')

import sys, importlib
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path: sys.path.insert(0, _p)
import degradation; importlib.reload(degradation)
from degradation import DegradedDataset, hf_transform, clean_tensor_transform
import cost; importlib.reload(cost)
from cost import data_cost
import env_config; importlib.reload(env_config)
print('Modules charges.')

In [ ]:
DATASET_NAME   = 'Imagewoof'
BASE_DIR       = env_config.ensure_dataset_ready(DATASET_NAME)
RESULTS_DIR    = env_config.results_dir(DATASET_NAME)
os.makedirs(RESULTS_DIR, exist_ok=True)

COST_HF        = 10
COST_BF        = 1
BATCH_SIZE     = 64
NUM_WORKERS    = min(8, os.cpu_count() or 2)
HF_TRAIN_RATIO = 0.8
SEEDS          = [42, 1, 2]
N_OPTUNA_TRIALS = 20
USE_WANDB      = True
WANDB_PROJECT  = 'PF22-MultiFidelity'

DEFAULT_PARAMS = {
    'lr': 1e-3, 'ep_per_phase': 5, 'ratio_mid1': 0.25, 'ratio_mid2': 0.60,
}

for req in ['train/BF', 'train/HF', 'test']:
    p = os.path.join(BASE_DIR, req)
    if not os.path.isdir(p): raise FileNotFoundError(f'Dossier manquant: {p}')

transform_hf    = hf_transform()
transform_clean = clean_tensor_transform()
dataset_hf_full  = datasets.ImageFolder(os.path.join(BASE_DIR, 'train/HF'), transform=transform_hf)
dataset_bf_train = DegradedDataset(datasets.ImageFolder(os.path.join(BASE_DIR, 'train/BF'), transform=transform_clean), seeded=True)
dataset_test_hf  = datasets.ImageFolder(os.path.join(BASE_DIR, 'test'), transform=transform_hf)
dataset_test_bf  = DegradedDataset(datasets.ImageFolder(os.path.join(BASE_DIR, 'test'), transform=transform_clean), seeded=True)

NUM_CLASSES = len(dataset_hf_full.classes)
print(f'Classes ({NUM_CLASSES}): {dataset_hf_full.classes}')

In [ ]:
class DomainLabeledConcatDataset(Dataset):
    def __init__(self, hf_dataset, bf_dataset):
        self.hf = hf_dataset; self.bf = bf_dataset; self.hf_len = len(hf_dataset)
    def __len__(self): return self.hf_len + len(self.bf)
    def __getitem__(self, idx):
        if idx < self.hf_len: x, y = self.hf[idx]; return x, y, 1
        x, y = self.bf[idx - self.hf_len]; return x, y, 0

class MixedBatchSampler(Sampler):
    def __init__(self, hf_indices, bf_indices, batch_size, hf_ratio, seed=42):
        self.hf_indices = list(hf_indices); self.bf_indices = list(bf_indices)
        self.batch_size = batch_size; self.seed = seed; self.epoch = 0
        self.hf_per_batch = max(1, int(round(batch_size * hf_ratio)))
        self.bf_per_batch = batch_size - self.hf_per_batch
        if self.bf_per_batch <= 0: raise ValueError('bf_per_batch <= 0')
        self.num_batches = min(len(self.hf_indices)//self.hf_per_batch,
                               len(self.bf_indices)//self.bf_per_batch)
        if self.num_batches <= 0: raise ValueError('Pas assez de donnees.')
    def __len__(self): return self.num_batches
    def set_epoch(self, e): self.epoch = e
    def __iter__(self):
        g = torch.Generator(); g.manual_seed(self.seed + self.epoch)
        hp = torch.randperm(len(self.hf_indices), generator=g).tolist()
        bp = torch.randperm(len(self.bf_indices), generator=g).tolist()
        for b in range(self.num_batches):
            batch = ([self.hf_indices[hp[b*self.hf_per_batch+i]] for i in range(self.hf_per_batch)] +
                     [self.bf_indices[bp[b*self.bf_per_batch+j]] for j in range(self.bf_per_batch)])
            order = torch.randperm(len(batch), generator=g).tolist()
            yield [batch[i] for i in order]

def create_model(num_classes=10):
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes); return m.to(device)

def evaluate(model, loader, criterion):
    model.eval(); total_loss = total_correct = total_n = 0
    with torch.no_grad():
        for batch in loader:
            x, y = batch[0].to(device), batch[1].to(device)
            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(x); loss = criterion(logits, y)
            total_loss += loss.item()*x.size(0)
            total_correct += (logits.argmax(1)==y).sum().item(); total_n += x.size(0)
    return total_loss/total_n, 100.0*total_correct/total_n

def make_grad_scaler():
    try:    return GradScaler('cuda', enabled=(device.type == 'cuda'))
    except TypeError: return GradScaler(enabled=(device.type == 'cuda'))

print('Helpers Curriculum definis.')

In [ ]:
def build_curriculum_loaders(hf_ds, bf_ds, schedule, seed=42):
    """Construit un dict {hf_ratio: DataLoader} pour chaque phase du schedule."""
    loaders = {}
    loaders[0.0] = DataLoader(bf_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
    loaders[1.0] = DataLoader(hf_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
    concat_ds = DomainLabeledConcatDataset(hf_ds, bf_ds)
    hf_idx = list(range(len(hf_ds)))
    bf_idx = list(range(len(hf_ds), len(hf_ds) + len(bf_ds)))
    for _, hf_ratio, _ in schedule:
        if 0.0 < hf_ratio < 1.0 and hf_ratio not in loaders:
            samp = MixedBatchSampler(hf_idx, bf_idx, BATCH_SIZE, hf_ratio, seed)
            loaders[hf_ratio] = DataLoader(concat_ds, batch_sampler=samp,
                                            num_workers=NUM_WORKERS, pin_memory=True)
    return loaders

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def run_with_params(seed, lr, ep_per_phase, ratio_mid1, ratio_mid2,
                     use_wandb=False, run_name_suffix=''):
    """Entraine Strategie 3 Curriculum avec hyperparametres explicites."""
    set_all_seeds(seed)
    schedule = [
        (ep_per_phase,   0.00, 'BF_pur'),
        (2*ep_per_phase, ratio_mid1, 'Mixte_A'),
        (3*ep_per_phase, ratio_mid2, 'Mixte_B'),
        (4*ep_per_phase, 1.00, 'HF_pur'),
    ]
    total_epochs = 4 * ep_per_phase
    n_hf = int(HF_TRAIN_RATIO * len(dataset_hf_full))
    ds_hf_train, ds_hf_val = random_split(
        dataset_hf_full, [n_hf, len(dataset_hf_full) - n_hf],
        generator=torch.Generator().manual_seed(seed))
    ld_val  = DataLoader(ds_hf_val, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
    ld_t_hf = DataLoader(dataset_test_hf, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
    ld_t_bf = DataLoader(dataset_test_bf, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
    cl_loaders = build_curriculum_loaders(ds_hf_train, dataset_bf_train, schedule, seed)

    def get_phase(epoch):
        for end_ep, hf_ratio, phase_name in schedule:
            if epoch <= end_ep: return hf_ratio, phase_name
        return schedule[-1][1], schedule[-1][2]

    track = use_wandb and _WANDB_AVAILABLE
    if track:
        wandb.init(project=WANDB_PROJECT,
                   name=f'S3_Optuna_{DATASET_NAME}_{run_name_suffix}_seed{seed}',
                   reinit=True, config={'lr': lr, 'ep_per_phase': ep_per_phase,
                       'ratio_mid1': ratio_mid1, 'ratio_mid2': ratio_mid2,
                       'dataset': DATASET_NAME, 'seed': seed})
    model = create_model(num_classes=NUM_CLASSES)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scaler = make_grad_scaler()
    cumulative_cost = 0; t0 = time.time()

    for epoch in range(1, total_epochs + 1):
        hf_ratio, phase_name = get_phase(epoch)
        loader = cl_loaders[hf_ratio]
        if hasattr(loader, 'batch_sampler') and hasattr(loader.batch_sampler, 'set_epoch'):
            loader.batch_sampler.set_epoch(epoch)
        model.train(); ep_loss = 0.0; ep_n = 0
        for batch in loader:
            x = batch[0].to(device, non_blocking=True)
            y = batch[1].to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(x); loss = criterion(logits, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            ep_loss += loss.item()*x.size(0); ep_n += x.size(0)
        train_loss = ep_loss / ep_n
        _, val_acc = evaluate(model, ld_val, criterion)
        print(f'[{phase_name}] Ep {epoch}/{total_epochs}: train={train_loss:.4f} val_acc={val_acc:.2f}%')
        if hf_ratio == 0.0:
            cumulative_cost += len(dataset_bf_train) * COST_BF
        elif hf_ratio == 1.0:
            cumulative_cost += len(ds_hf_train) * COST_HF
        else:
            s = loader.batch_sampler
            cumulative_cost += s.num_batches*(s.hf_per_batch*COST_HF + s.bf_per_batch*COST_BF)
        if track:
            wandb.log({'epoch': epoch, 'phase': phase_name,
                       'train/loss': train_loss, 'val/accuracy': val_acc})

    tmin = (time.time() - t0) / 60.0
    crit = nn.CrossEntropyLoss()
    _, val_acc_final = evaluate(model, ld_val,   crit)
    _, hf_acc        = evaluate(model, ld_t_hf,  crit)
    _, bf_acc        = evaluate(model, ld_t_bf,  crit)
    mix_acc = (hf_acc + bf_acc) / 2.0
    if track:
        wandb.log({'test/accuracy_HF': hf_acc, 'test/accuracy_BF': bf_acc}); wandb.finish()
    print(f'[seed {seed}] HF={hf_acc:.2f}% BF={bf_acc:.2f}% | {tmin:.1f} min')
    return {'val_acc': val_acc_final, 'accuracy_HF': hf_acc, 'accuracy_BF': bf_acc,
            'accuracy_Mixte': mix_acc, 'total_cost_CA': float(cumulative_cost),
            'time_min': tmin, 'schedule': schedule, 'model': model}

def objective(trial):
    lr          = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    ep_per_phase = trial.suggest_int('ep_per_phase', 3, 8)
    ratio_mid1  = trial.suggest_float('ratio_mid1', 0.10, 0.40)
    ratio_mid2  = trial.suggest_float('ratio_mid2', 0.45, 0.85)
    r = run_with_params(42, lr, ep_per_phase, ratio_mid1, ratio_mid2)
    return r['val_acc']

print('Fonctions run_with_params et objective definies.')

In [ ]:
if not _OPTUNA_AVAILABLE:
    raise RuntimeError('Optuna non installe. Executer: pip install optuna')

sampler_opt = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler_opt,
                            study_name=f'strat3_{DATASET_NAME}')

print(f'Lancement de {N_OPTUNA_TRIALS} trials Optuna (Strategy 3, {DATASET_NAME})...')
t0_opt = time.time()
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)
t_optuna_min = (time.time() - t0_opt) / 60.0

best_params = study.best_params
best_trial  = study.best_trial
print(f'Optuna termine en {t_optuna_min:.1f} min')
print(f'Best trial #{best_trial.number} | val_acc={best_trial.value:.2f}%')
print(f'Best params: {best_params}')

all_trials = [{
    'number': t.number, 'value': t.value if t.value is not None else float('nan'),
    'params': t.params, 'state': str(t.state)
} for t in study.trials]

optuna_results = {
    'strategy': 'curriculum_learning_progressif', 'dataset': DATASET_NAME,
    'n_trials': N_OPTUNA_TRIALS, 'optimization_seed': 42,
    'best_trial_number': best_trial.number,
    'best_val_acc_hf_pct': float(best_trial.value),
    'default_hyperparams': DEFAULT_PARAMS, 'best_params': best_params,
    'optuna_time_min': t_optuna_min, 'all_trials': all_trials,
}
params_path = os.path.join(RESULTS_DIR, 'results_strategy3_optuna_best_params.json')
with open(params_path, 'w', encoding='utf-8') as f:
    json.dump(optuna_results, f, indent=2, ensure_ascii=False)
print('Best params JSON:', params_path)

In [ ]:
print(f'Evaluation finale (3 seeds) avec best params: {best_params}')
t0_final = time.time()
per_seed_opt = [
    run_with_params(s, lr=best_params['lr'], ep_per_phase=best_params['ep_per_phase'],
                    ratio_mid1=best_params['ratio_mid1'], ratio_mid2=best_params['ratio_mid2'],
                    use_wandb=USE_WANDB, run_name_suffix='optimized')
    for s in SEEDS
]
model_opt = per_seed_opt[0]['model']

def _agg(key):
    vals = [r[key] for r in per_seed_opt]
    m = statistics.mean(vals); s = statistics.stdev(vals) if len(vals) > 1 else 0.0
    return m, s, vals

hf_m, hf_s, hf_all = _agg('accuracy_HF')
bf_m, bf_s, bf_all = _agg('accuracy_BF')
mx_m, mx_s, mx_all = _agg('accuracy_Mixte')
tt_m, tt_s, _      = _agg('time_min')
cost_m, _, _       = _agg('total_cost_CA')

n_hf_pool = len(dataset_hf_full); n_bf_pool = len(dataset_bf_train)
data_cost_CA = data_cost(n_hf=n_hf_pool, n_bf=n_bf_pool)

best_schedule = per_seed_opt[0]['schedule']
optimized_results = {
    'strategy': 'curriculum_learning_progressif', 'dataset': DATASET_NAME,
    'hyperparams_source': 'optuna_optimized', 'hyperparams': best_params,
    'default_hyperparams': DEFAULT_PARAMS, 'n_optuna_trials': N_OPTUNA_TRIALS,
    'best_optuna_val_acc': float(best_trial.value),
    'best_schedule': [(e, r, p) for e, r, p in best_schedule],
    'total_epochs': 4 * best_params['ep_per_phase'],
    'seeds': SEEDS, 'n_hf_pool': n_hf_pool, 'n_bf_pool': n_bf_pool,
    'data_cost_CA': float(data_cost_CA), 'total_cost_CA': cost_m,
    'accuracy_HF': hf_m, 'accuracy_HF_std': hf_s, 'accuracy_HF_seeds': hf_all,
    'accuracy_BF': bf_m, 'accuracy_BF_std': bf_s, 'accuracy_BF_seeds': bf_all,
    'accuracy_Mixte': mx_m, 'accuracy_Mixte_std': mx_s, 'accuracy_Mixte_seeds': mx_all,
    'total_time_min': tt_m, 'total_time_min_std': tt_s,
    'per_seed': [{k: v for k, v in r.items() if k != 'model'} for r in per_seed_opt],
}

default_path = os.path.join(RESULTS_DIR, 'results_strategy3_curriculum_learning.json')
if os.path.exists(default_path):
    with open(default_path, 'r', encoding='utf-8') as f: def_r = json.load(f)
    optimized_results['delta_accuracy_HF']    = hf_m - def_r.get('accuracy_HF', 0)
    optimized_results['delta_accuracy_BF']    = bf_m - def_r.get('accuracy_BF', 0)
    optimized_results['delta_accuracy_Mixte'] = mx_m - def_r.get('accuracy_Mixte', 0)

opt_path = os.path.join(RESULTS_DIR, 'results_strategy3_curriculum_learning_optimized.json')
pth_path = os.path.join(RESULTS_DIR, 'model_strategy3_curriculum_learning_optimized.pth')
with open(opt_path, 'w', encoding='utf-8') as f:
    json.dump(optimized_results, f, indent=2, ensure_ascii=False)
torch.save(model_opt.state_dict(), pth_path)

print(f'--- RESULTATS OPTIMISES - Strategy 3 - {DATASET_NAME} ---')
print(f'Test HF    : {hf_m:.2f} +/- {hf_s:.2f} %')
print(f'Test BF    : {bf_m:.2f} +/- {bf_s:.2f} %')
print(f'Test Mixte : {mx_m:.2f} +/- {mx_s:.2f} %')
if 'delta_accuracy_HF' in optimized_results:
    print(f'Delta HF   : {optimized_results["delta_accuracy_HF"]:+.2f}%')
print('JSON:', opt_path)

In [ ]:
trial_vals = [t['value'] for t in all_trials if t['value'] == t['value']]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Strategie 3 — Optuna Curriculum ({DATASET_NAME}, {N_OPTUNA_TRIALS} trials)', fontsize=14)

axes[0].plot(range(1, len(trial_vals)+1), trial_vals, marker='o', markersize=4,
             linewidth=1.5, color='tab:blue')
axes[0].axhline(max(trial_vals), color='tab:red', linestyle='--',
                label=f'Best: {max(trial_vals):.2f}%')
axes[0].set_title('Progression des trials Optuna')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Val acc HF (%)')
axes[0].legend(); axes[0].grid(alpha=0.3)

default_path = os.path.join(RESULTS_DIR, 'results_strategy3_curriculum_learning.json')
if os.path.exists(default_path):
    with open(default_path, 'r', encoding='utf-8') as f: def_r = json.load(f)
    metrics = ['Test HF', 'Test BF', 'Test Mixte']
    def_vals = [def_r.get('accuracy_HF',0), def_r.get('accuracy_BF',0), def_r.get('accuracy_Mixte',0)]
    def_stds = [def_r.get('accuracy_HF_std',0), def_r.get('accuracy_BF_std',0), def_r.get('accuracy_Mixte_std',0)]
    x = np.arange(len(metrics)); w = 0.3
    b1 = axes[1].bar(x - w/2, def_vals, w, label='Default', color='#64B5CD',
                     yerr=def_stds, capsize=4, alpha=0.85)
    b2 = axes[1].bar(x + w/2, [hf_m, bf_m, mx_m], w, label='Optimise (Optuna)',
                     color='#E07070', yerr=[hf_s, bf_s, mx_s], capsize=4, alpha=0.85)
    for bars, vals in [(b1, def_vals), (b2, [hf_m, bf_m, mx_m])]:
        for bar, v in zip(bars, vals):
            axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
                        f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
    axes[1].set_title('Precisions: Default vs Optimise')
    axes[1].set_xticks(x); axes[1].set_xticklabels(metrics)
    axes[1].set_ylim(0, 105); axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'Resultats default non trouves.',
                ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout(rect=[0, 0, 1, 0.95])
fig_path = os.path.join(RESULTS_DIR, 'strategy3_optuna_comparison.png')
plt.savefig(fig_path, dpi=180, bbox_inches='tight'); plt.show()
print('Figure:', fig_path)

In [ ]:
print(f'\n=== HYPERPARAMETRES: Default vs Optuna (Strategie 3) ===')
param_names = list(best_params.keys())
hdr = f"{'Parametre':<20} {'Defaut':>15} {'Optuna':>15}"
print(hdr); print('-'*len(hdr))
for k in param_names:
    dv = DEFAULT_PARAMS.get(k, 'N/A')
    ov = best_params[k]
    dv_str = f'{dv:.2e}' if isinstance(dv, float) else str(dv)
    ov_str = f'{ov:.2e}' if isinstance(ov, float) else str(ov)
    print(f'{k:<20} {dv_str:>15} {ov_str:>15}')
print(f"Total epochs default: {4*DEFAULT_PARAMS['ep_per_phase']} | Optuna: {4*best_params['ep_per_phase']}")

comp_files = {
    'BL 1(HF)':             'results_baseline_HF.json',
    'BL 2(BF)':             'results_baseline_BF.json',
    'BL 3(MIXTE)':          'results_baseline_MIXTE.json',
    'Strat 1(default)':     'results_strategy1_transfer_learning.json',
    'Strat 1(Optuna)':      'results_strategy1_transfer_learning_optimized.json',
    'Strat 2(default)':     'results_strategy2_cotraining_reweighting.json',
    'Strat 2(Optuna)':      'results_strategy2_cotraining_reweighting_optimized.json',
    'Strat 3(default)':     'results_strategy3_curriculum_learning.json',
    'Strat 3(Optuna)':      'results_strategy3_curriculum_learning_optimized.json',
    'Strat 5(default)':     'results_strategy5_ewc.json',
    'Strat 5(Optuna)':      'results_strategy5_ewc_optimized.json',
}
loaded = {}
for name, fn in comp_files.items():
    p = os.path.join(RESULTS_DIR, fn)
    if os.path.exists(p):
        with open(p, 'r', encoding='utf-8') as f: loaded[name] = json.load(f)

if loaded:
    print(f'\n=== RECAPITULATIF COMPLET — {DATASET_NAME} ===')
    hdr = f"{'Modele':<30} {'HF%':>7} {'BF%':>7} {'Mixte%':>9} {'Cout(CA)':>11}"
    print(hdr); print('-'*len(hdr))
    for name, d in loaded.items():
        c = int(d.get('total_cost_CA', 0))
        print(f"{name:<30} {d.get('accuracy_HF', float('nan')):>7.2f} "
              f"{d.get('accuracy_BF', float('nan')):>7.2f} "
              f"{d.get('accuracy_Mixte', float('nan')):>9.2f} {c:>11}")